# Sokoban MCTS + LLM Optimization Pipeline

All configuration comes from two files in `MCTS_tools/`:
1. **`hyperparams/default_hyperparams.py`** — engine params (iterations,
   max_rollout_depth, exploration_weight), game identity, and optimization
   settings (phases, num_iters, etc.)
2. **`training_logic/sokoban_training.py`** — levels, mastery criteria,
   level-selection strategy

The orchestrator (`OptimizationRunner.from_config()`) reads both files
and drives the iterative LLM optimization loop.

## Cell 1: Setup — add Tool_Creation to sys.path & import orchestrator

In [1]:
import sys
from pathlib import Path

# Ensure Tool_Creation is on sys.path (parent of orchestrator/)
ROOT = Path(".").resolve().parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from orchestrator import OptimizationRunner, Evaluator
from mcts import MCTSEngine
from mcts.games import Sokoban

print(f"Working dir: {Path('.').resolve()}")
print(f"Tool_Creation root: {ROOT}")
print("All imports OK ✓")

Working dir: /Users/zhuyuezx/Documents/UCSD/Winter_2025/CSE291A/CSE291A_Project/Tool_Creation/orchestrator
Tool_Creation root: /Users/zhuyuezx/Documents/UCSD/Winter_2025/CSE291A/CSE291A_Project/Tool_Creation
All imports OK ✓


## Cell 2: Play one baseline game (sanity check)

Quick check that the engine works before launching the optimization loop.

In [2]:
import importlib.util

# Load hyperparams module (single source of config)
_hp_path = ROOT / "MCTS_tools" / "hyperparams" / "default_hyperparams.py"
_spec = importlib.util.spec_from_file_location("hp", str(_hp_path))
_mod = importlib.util.module_from_spec(_spec)
_spec.loader.exec_module(_mod)
hp = _mod.get_hyperparams()

ITERS = hp["iterations"]
MAX_DEPTH = hp["max_rollout_depth"]
MAX_STEPS = _mod.CONSTRUCTOR_KWARGS["max_steps"]
PHASES = _mod.PHASES

# Load training logic to get start level
_tl_path = ROOT / "MCTS_tools" / "training_logic" / f"{_mod.TRAINING_LOGIC}.py"
_spec2 = importlib.util.spec_from_file_location("tl", str(_tl_path))
_tl_mod = importlib.util.module_from_spec(_spec2)
_spec2.loader.exec_module(_tl_mod)
LEVEL = _tl_mod.START_LEVEL

game = Sokoban(LEVEL, max_steps=MAX_STEPS)
engine = MCTSEngine(game, iterations=ITERS, max_rollout_depth=MAX_DEPTH, logging=True)
baseline = engine.play_game()

base_tag = "SOLVED" if baseline.get("solved") else "UNSOLVED"
print(f"Baseline ({LEVEL}): {base_tag} in {baseline.get('steps', '?')} steps  "
      f"returns={baseline.get('returns', '?')}")
print(f"Hyperparams: iterations={ITERS}, max_depth={MAX_DEPTH}, "
      f"C={hp.get('exploration_weight', 1.41)}")
print(f"Optimize phases: {PHASES}")
print(f"Trace: {baseline.get('log_file', 'N/A')}")

Baseline (level6): UNSOLVED in 200 steps  returns=[0.0]
Hyperparams: iterations=200, max_depth=500, C=1.41
Optimize phases: ['simulation', 'hyperparams', 'expansion']
Trace: /Users/zhuyuezx/Documents/UCSD/Winter_2025/CSE291A/CSE291A_Project/Tool_Creation/mcts/records/Sokoban (level6)_20260306_234316_390406.json


## Cell 3: Run iterative optimization

Create the `OptimizationRunner` from the hyperparams module and run the
full loop. Engine params, game identity, and optimization settings all
come from `MCTS_tools/hyperparams/default_hyperparams.py`. Training
strategy comes from `MCTS_tools/training_logic/`.

In [3]:
runner = OptimizationRunner.from_config(verbose=True)
summary = runner.run()

best_fns = summary["best_fns"]
print(f"\nbest_fns: { {p: ('set' if f else 'None') for p, f in best_fns.items()} }")
print(f"Final hyperparams: {summary.get('current_hyperparams', {})}")

Starting level: level6
Hyperparams: iterations=200, max_depth=500, C=1.410
Phases to optimize: ['simulation', 'hyperparams', 'expansion']
  Computing baseline for level6…
    level6: composite=0.0000, solve_rate=0%, avg_returns=0.0000 (3.8s)
  Reject floor for level6: 0.0000
  Active levels: ['level1', 'level2', 'level3', 'level4', 'level5', 'level6', 'level7', 'level8', 'level9', 'level10']

############################################################
  ITERATION 1/5, LEVEL=level6, PHASE=expansion
  Baseline composite=0.0000, reject_floor=0.0000
  Active levels: 10/10, mastered: []
############################################################
  Play: UNSOLVED in 200 steps  returns=0.0000  (1.2s)
Step 1/6: Building analysis prompt…
Step 2/6: Querying LLM (step 2 — draft code)…
Step 3/6: Querying LLM (step 3 — critique & finalize)…
  LLM query: status=success  elapsed=33.86s
  Step-1 analysis length: 4016 chars
Step 5/6: Parsing & validating response…
  Validation passed ✓
  Installed to

## Cell 4: Multi-Level Evaluation

Run both **baseline** (default MCTS) and **best LLM-optimized** function
across all levels to measure generalization.

In [5]:
best_fns = summary["best_fns"]
levels   = summary["active_levels"] + list(summary.get("mastered_levels", []))
ev       = runner.evaluator

# Build extra_tools from best_fns for optimized eval
opt_tools = {p: f for p, f in best_fns.items() if f is not None} or None

has_optimized = opt_tools is not None
print(f"Best fns: { {p: ('set' if f else 'None') for p, f in best_fns.items()} }")
print(f"Final hyperparams: {summary.get('current_hyperparams', {})}")
print(f"Mastered: {sorted(summary.get('mastered_levels', set()))}")
print(f"Level best scores: {summary.get('level_best_scores', {})}")

if not has_optimized:
    print("\nNo optimized functions adopted — skipping comparative eval.")
else:
    # Only eval on levels visited during optimization (have baselines)
    eval_levels = sorted(ev.level_baselines.keys())
    print(f"\nEvaluating {len(eval_levels)} levels (n=3 each)…")

    rows = []
    for lvl in eval_levels:
        avg_b, sr_b, steps_b, _, t_b = ev.multi_eval(None, lvl, n=3, logging=False)
        avg_o, sr_o, steps_o, _, t_o = ev.multi_eval(
            None, lvl, n=3, logging=False, extra_tools=opt_tools,
        )
        rows.append((lvl, sr_b, sr_o, avg_b, avg_o, steps_b, steps_o, t_b, t_o))
        print(f"{lvl}: baseline={avg_b:.3f} ({sr_b:.0%})  optimized={avg_o:.3f} ({sr_o:.0%})  "
              f"[{t_b:.1f}s + {t_o:.1f}s]")

    print(f"\n{'Level':<10} {'Base Solve%':>12} {'Opt Solve%':>12} "
          f"{'Base AvgRet':>12} {'Opt AvgRet':>12} "
          f"{'Base Steps':>11} {'Opt Steps':>11}")
    print("─" * 82)
    for lvl, sr_b, sr_o, avg_b, avg_o, steps_b, steps_o, *_ in rows:
        print(f"{lvl:<10} {sr_b*100:>11.0f}% {sr_o*100:>11.0f}% "
              f"{avg_b:>12.3f} {avg_o:>12.3f} "
              f"{steps_b:>11.1f} {steps_o:>11.1f}")

Best fns: {'simulation': 'set', 'expansion': 'None'}
Final hyperparams: {'iterations': 1000, 'max_rollout_depth': 130, 'exploration_weight': 2.0}
Mastered: ['level2', 'level4', 'level5']
Level best scores: {'level6': 0.0, 'level4': 1.0, 'level2': 1.0, 'level5': 1.0, 'level10': 0.6666666666666666}

Evaluating 5 levels (n=3 each)…
level10: baseline=0.000 (0%)  optimized=0.333 (33%)  [57.2s + 479.8s]
level2: baseline=1.000 (100%)  optimized=1.000 (100%)  [0.1s + 0.4s]
level4: baseline=1.000 (100%)  optimized=1.000 (100%)  [0.5s + 4.6s]
level5: baseline=1.000 (100%)  optimized=0.667 (67%)  [24.1s + 383.8s]
level6: baseline=0.000 (0%)  optimized=0.000 (0%)  [18.5s + 27.4s]

Level       Base Solve%   Opt Solve%  Base AvgRet   Opt AvgRet  Base Steps   Opt Steps
──────────────────────────────────────────────────────────────────────────────────
level10              0%          33%        0.000        0.333       200.0       196.0
level2             100%         100%        1.000        1.000   